In [1]:
import pandas as pd
import googlemaps
import folium
import warnings
import os
from folium.plugins import MarkerCluster

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel("../AliancaCentro-Rio/Aliança Centro-Rio 3.0.xlsx")
latlongs = pd.read_excel("../AliancaCentro-Rio/8. LATLONGS/data/ocorrencias.xlsx")

In [3]:
df.drop(columns=["Latitude", "Longitude"], inplace=True)

In [4]:
df = pd.merge(df,
              latlongs[["Número do protocolo", "Latitude", "Longitude"]].drop_duplicates(), 
              how="left", 
              on="Número do protocolo")

In [5]:
df.drop_duplicates(subset="Número do protocolo", keep="first", inplace=True)

In [6]:
df.head()

,Grau de risco,Semana,Agente,Mês,Data,Turno,Número do protocolo,Status,Resposta do 1746,Parecer,...,Região,Categoria,Serviços,Tipo de ocorrência,Número de pessoas,Descrição,Observação,Foto Solucionado (dummy),Latitude,Longitude
0,3,1,Mylena,set./21,2021-09-02 00:00:00,Tarde,23708192-8,Fechado,Nao foi localizado buraco no endereço informad...,Solucionado,...,1,Zeladoria,Conservação,Recapeamento / Recuperação de pavimentação,NaN,Buracos na calçada por falta de pedras portugu...,NaN,1,-22.901312,-43.178077
1,3,1,Mylena,set./21,2021-09-02 00:00:00,Tarde,23721772-0,Fechado,A Equipe Especializada em Abordagem Social da ...,Rejeição de Assistência,...,2,Moradores de Rua,Assistência Social,Atendimento a pessoas em situação de rua,4.0,Pessoas em situação de rua.,"Como o protocolo foi aberto há muito tempo, é ...",NaN,-22.906148,-43.177039
2,3,1,Mylena,set./21,2021-09-02 00:00:00,Tarde,23721752-2,Fechado,Foi emitida uma notificação para o responsável...,Solucionado,...,2,Zeladoria,Conservação,Fiscalização de buraco na calçada,NaN,Buraco na calçada. Risco de acidente,Monitoramento no dia 04/10 verificou que o pro...,NaN,-22.905745,-43.177271
3,3,1,Mylena,set./21,2021-09-02 00:00:00,Tarde,23708871-5,Fechado,Informamos que o serviço foi executado.,Solucionado,...,2,Zeladoria,Conservação,Recapeamento / Recuperação de pavimentação,NaN,Pedras soltas,NaN,NaN,-22.902184,-43.178542
4,3,1,Mylena,set./21,2021-09-02 00:00:00,Tarde,23822143-0,Fechado,Informamos que a sua solicitação foi incluída ...,Redirecionado para SICOP,...,2,Ambulantes,Ordem Pública,Fiscalização de comércio ambulante,3.0,Vendedores ambulantes irregulares na calçada,É improvável que estejam no mesmo local,NaN,-22.906148,-43.177039


In [7]:
df["Parecer"].fillna("-", inplace=True)
df["Status 2"].fillna("-", inplace=True)

In [8]:
tipo = []
for i, row in df.iterrows():
    if row["Status"] == "Fechado" and row["Status 2"] == "Em andamento":
        tipo.append("Em andamento")
    elif row["Status"] == "Em andamento":
        tipo.append("Em andamento")
    elif row["Parecer"] == "Solucionado" or row["Parecer 2"] == "Solucionado":
        tipo.append("Solucionado")
    else:
        tipo.append("Fechado por outros motivos")

df["Tipo"] = tipo

In [9]:
# Create the base map
mapa = folium.Map(location=[-22.900252, -43.178084], zoom_start=20, tiles='CartoDB Positron')

# Add title
title_html = '''
             <h3 align="center" style="font-size:20px"><b>Mapa geral com todas as ocorrências</b></h3>
             '''
mapa.get_root().html.add_child(folium.Element(title_html))

# Create separate marker clusters for each "tipo"
marker_cluster_em_andamento = MarkerCluster(name='Em andamento').add_to(mapa)
marker_cluster_solucionado = MarkerCluster(name='Solucionado').add_to(mapa)
marker_cluster_fechado = MarkerCluster(name='Fechado por outros motivos').add_to(mapa)

# Iterate through the DataFrame and add markers to the appropriate clusters
for i, row in df.iterrows():
    
    # if row['Número do protocolo'] in ["23822232-7", "26284880-0", "23766042-0", "23766027-5", "23907298-2", 
    #                                   "23907325-7", "24011158-5", "24063104-8", "26020744-1", "26227307-2", 
    #                                   "26596015-0", "27706615-0", "23766068-0", "26596027-2", "26681648-5",
    #                                   "25461271-8", "29132523-8", "29333121-8", "29471891-6"]:
    #     continue
    
    protocolo = row['Número do protocolo']
    endereco = row['Endereço']
    ponto_referencia = row['Ponto de referência']
    regiao = row['Região']
    status1 = row['Status']
    parecer = row['Parecer']
    status2 = row['Status 2']
    lat = row['Latitude']
    lng = row['Longitude']
    tipo = row["Tipo"]

    string = f"""
    Protocolo = {protocolo}<br>
    Endereço = {endereco}<br>
    Ponto de referência = {ponto_referencia}<br>
    Região = {regiao}<br>
    Status 1 = {status1}<br>
    Parecer = {parecer}<br>
    Status 2 = {status2}<br>
    Tipo = {tipo}<br>
    """
    
    if os.path.isfile(f'fotos/{protocolo}.jpeg'):
        string += f"<img src='fotos/{protocolo}.jpeg'>"
    elif os.path.isfile(f'fotos/{protocolo}.jpg'):
        string += f"<img src='fotos/{protocolo}.jpg'>"
    elif os.path.isfile(f'fotos/{protocolo}.JPEG'):
        string += f"<img src='fotos/{protocolo}.JPEG'>"
    elif os.path.isfile(f'fotos/{protocolo}.JPG'):
        string += f"<img src='fotos/{protocolo}.JPG'>"

    popup = folium.Popup(string, max_width=300, min_width=300)
    marker = folium.Marker([lat, lng], popup=popup, icon=folium.Icon(color='orange' if tipo == "Em andamento" else 'blue' if tipo == "Solucionado" else 'gray'))

    # Add marker to the appropriate cluster based on "tipo"
    if tipo == "Em andamento":
        marker.add_to(marker_cluster_em_andamento)
    elif tipo == "Solucionado":
        marker.add_to(marker_cluster_solucionado)
    elif tipo == "Fechado por outros motivos":
        marker.add_to(marker_cluster_fechado)

# Add layer control to the map
folium.LayerControl().add_to(mapa)

# Save the map to an HTML file
mapa.save('index.html')

In [10]:
mapa